# Gemma 4 E4B — Text-to-SQL Fine-Tuning on BIRD

**Run this notebook on Google Colab with T4 GPU**

Runtime → Change runtime type → T4 GPU

This notebook runs all 6 steps in sequence.

**Paths:** On your machine, if the BIRD train bundle lives under `Data/train/train/` (with `train.json` and `train_databases/`), the project uses it automatically. On Colab, Step 1 downloads into `data/bird/` instead.

In [14]:
import os
repo_dir = "/content/Finetuned_Gemma4_t2sql"
os.chdir(repo_dir)
!git pull origin main

remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 21 (delta 13), reused 20 (delta 12), pack-reused 0 (from 0)
Unpacking objects: 100% (21/21), 7.08 KiB | 805.00 KiB/s, done.
From https://github.com/KARAN6550/Finetuned_Gemma4_t2sql
 * branch            main       -> FETCH_HEAD
   f524b03..bb37511  main       -> origin/main
Updating f524b03..bb37511
Fast-forward
 colab_run_all.ipynb           | 360 ++++++++++++++++++++++++------------------
 configs/training_config.py    |  77 ++++++++-
 scripts/01_download_bird.py   |  95 +++++++++--
 scripts/02_extract_schemas.py |  20 +--
 scripts/05_evaluate.py        |  31 ++--
 5 files changed, 383 insertions(+), 200 deletions(-)


In [15]:
# ── Cell 1: Clone your repo and install dependencies ─────────────────────────
# Replace with your GitHub repo URL after you push the project

!git clone https://github.com/KARAN6550/Finetuned_Gemma4_t2sql.git
%cd Finetuned_Gemma4_t2sql
!pip install -q -r requirements.txt

# Gemma 4 requires transformers >= 5.5.0 (model_type=gemma4 first appeared there).
# The line below is a safety net: if pip resolved an older version (e.g. due to
# a conflicting dep), this upgrades to the latest from source and is a no-op
# when 5.5.0+ is already installed.
import importlib, subprocess, sys
_tv = importlib.import_module("transformers").__version__
_major, _minor = (int(x) for x in _tv.split(".")[:2])
if (_major, _minor) < (5, 5):
    print(f"transformers {_tv} is too old for Gemma 4 — installing from source...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/huggingface/transformers.git"])
else:
    print(f"transformers {_tv} ✓")

fatal: destination path 'Finetuned_Gemma4_t2sql' already exists and is not an empty directory.
/content/Finetuned_Gemma4_t2sql/Finetuned_Gemma4_t2sql


In [16]:
# ── Cell 2: Mount Google Drive (to save checkpoints — free Colab disconnects) ─
from google.colab import drive
drive.mount('/content/drive')

import os
# Update checkpoint dir to save to Drive
os.environ['CHECKPOINT_DIR'] = '/content/drive/MyDrive/gemma4-text2sql/checkpoints'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
# ── Cell 3: Verify GPU ───────────────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

GPU: Tesla T4
VRAM: 15.6 GB


In [18]:
# ── Cell 4: STEP 1 — Download BIRD dataset ───────────────────────────────────
!python scripts/01_download_bird.py

  STEP 1: Downloading BIRD Dataset

[1/3] Downloading BIRD training split (~3.5 GB)...
  Downloading: https://bird-bench.oss-cn-beijing.aliyuncs.com/train.zip
  Saving to:   /content/Finetuned_Gemma4_t2sql/Finetuned_Gemma4_t2sql/data/bird/train.zip
  Progress: 100%  (8919.5 / 8919.5 MB).5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)344.9 / 8919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)742.1 / 8919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)667.7 / 8919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)221.1 / 8919.5 MB)919.5 MB)919.5 MB)919.5 MB)804.6 / 8919.5 MB)919.5 MB)976.5 / 8919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)889.5 / 8919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)377.8 / 8919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)797.4 / 8919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5 MB)919.5

In [19]:
# ── Cell 5: STEP 2 — Extract schemas from .sqlite files ──────────────────────
!python scripts/02_extract_schemas.py

  STEP 2: Extracting Database Schemas

  Found existing cache with 80 schemas.
  Total unique databases to process: 80
Extracting schemas: 100% 80/80 [00:00<00:00, 1185668.98it/s]

── Summary ──────────────────────────────────────────────────
  Total databases in cache : 80
  Newly extracted          : 0
  Failed (not found)       : 0

── Sample schema for 'address' (first 400 chars) ───────
-- Schema unavailable for database: address
  ...

  Schema cache saved to: /content/Finetuned_Gemma4_t2sql/Finetuned_Gemma4_t2sql/data/schemas.json

✓ STEP 2 COMPLETE — Run scripts/03_prepare_dataset.py next.



In [20]:
# ── Cell 6: STEP 3 — Format dataset into training prompts ────────────────────
!python scripts/03_prepare_dataset.py

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
  STEP 3: Preparing Training and Dev Datasets

[1/4] Loading BIRD JSON files and schema cache...
  Loaded 6,601 training examples
  Loaded 1,534 dev examples
  Loaded 80 database schemas

[2/4] Formatting examples into prompt template...
  Train: 100% 6601/6601 [00:00<00:00, 223827.78it/s]
  Dev  : 100% 1534/1534 [00:00<00:00, 208397.43it/s]

── Sample Formatted Prompt (example 0) ──────────────────────
### Task
Generate a valid SQL query to answer the following question using the database schema below.

### Database Schema
-- Schema unavailable for database: movie_platform

### Evidence
Sex, Drink and Bloodshed refers to movie title = 'Sex, Drink and Bloodshed';

### Question
Who is the director of the movie Sex, Drink and Bloodshed?

###

In [ ]:
# ── Cell 7: Login to W&B (run once) ──────────────────────────────────────────
# Get your API key from https://wandb.ai/authorize
import wandb
wandb.login()

In [ ]:
# ── Cell 8: STEP 4 — Train (4–6 hours) ───────────────────────────────────────
# Watch your W&B dashboard at https://wandb.ai while this runs
!python scripts/04_train.py

In [ ]:
# ── Cell 9: STEP 5 — Quick eval sanity check (100 examples) ──────────────────
!python scripts/05_evaluate.py --subset 100

In [ ]:
# ── Cell 10: STEP 5 — Full evaluation (1,534 examples, ~1 hour) ──────────────
!python scripts/05_evaluate.py

In [ ]:
# ── Cell 11: STEP 6 — Push to HuggingFace ────────────────────────────────────
# Set your HF token first:
import os
os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'   # get from huggingface.co/settings/tokens
!python scripts/06_push_to_hub.py